# DQI: exact 42-AND arithmetic, verified end to end

**21.18% lower DQI cost, with full official verification.** Mean Toffoli equivalents fall from **5.725 million to 4.503 million**; peak qubits change from **1,893 to 1,895**. Both circuits pass **102,400 shots with zero value or phase errors**.

This CPU-only notebook replaces 63-product field multiplication with an explicitly derived 42-product construction, carries the saving through inversion and division, and folds Frobenius maps into parity masks to avoid extra workspace. The supplied Reed-Solomon decoder, error distribution and stopping rule stay the same.

It generates and tests the arithmetic, builds the full RS(4095, 4025) decoder, runs the official DQI verifier, and writes a complete **submission.zip**. Fusion and chemistry use the official reference circuits. The projected three-task score is **1.082585** from this public-salt validation; it is not an observed leaderboard score.

**Run:** attach the competition data, select CPU, and Run All. No installation, internet, GPU or chemistry starter kit is needed. Both full-strength DQI verification runs are enabled by default. This run builds the candidate in about 153 seconds, then spends about 424 seconds verifying the two circuits.

Sources: [competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation), [organizers' paper](https://arxiv.org/abs/2610.23832), [DQI decoder paper](https://arxiv.org/abs/2510.10967), [Kickmix](https://github.com/quantumlib/Qualtran/tree/main/kickmix). Related rank-42 work is credited at the end. The organizers supplied the decoder and verifiers; the contribution here is the derived tensor, arithmetic adapter and reproducible validation.

In [ ]:
import sys, os, json, time, hashlib, platform, zipfile, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

COMPETITION = next(p.parent for p in Path('/kaggle/input').rglob('verify_submission_dqi.py')
                   if p.parent.name == 'dqi').parent.parent
# The file is COMPETITION/verifiers/dqi/verify_submission_dqi.py.
VERIFIERS = COMPETITION / 'verifiers'
SAMPLE = COMPETITION / 'sample_submission'
sys.path[:0] = [str(COMPETITION/'kickmix'), str(VERIFIERS)]
import kickmix as km
from dqi import dqi_reference_circuit as reference
from dqi import verify_submission_dqi as official

OUT = Path('/kaggle/working')
SCRATCH = Path('/tmp/tower42-build')
SCRATCH.mkdir(exist_ok=True)
SHOTS = 102_400
WORKERS = min(4, os.cpu_count() or 1)
PUBLIC_SALT = 'tower42-public-validation-v1'
field = km.GF2Field(12)
print({'python':platform.python_version(), 'numpy':np.__version__,
       'field_modulus':hex(field.modulus), 'shots_per_circuit':SHOTS,
       'workers':WORKERS, 'competition':str(COMPETITION)})

## 1. Optimize the cost the competition actually charges

For DQI the cost is $R(\epsilon^+) Q^2 T$. Fewer arithmetic products are useful only if the scratch space and failure overhead do not erase the saving. In particular, a local gate-count improvement is not a full-decoder result.

The comparison below keeps the official iteration count and decoder logic. It reports both Toffolis and peak qubits, verifies scratch cleanup and phases, and includes the official confidence-bound penalty. The final competition score is the cube root of the DQI improvement when the other two tasks use their reference circuits.

## 2. Where 42 products come from

Use the isomorphism

$$\mathrm{GF}(2^{12}) \cong \mathrm{GF}(8)[v]/(v^4+v+1), \qquad
\mathrm{GF}(8)=\mathrm{GF}(2)[u]/(u^3+u+1).$$

Each input becomes a degree-three polynomial in $v$. Its degree-six product is determined by six finite evaluation points plus infinity: seven products in GF(8). Each GF(8) product needs three diagonal products and three pairwise products $(x_i+x_j)(y_i+y_j)$. Thus $7\times6=42$ binary products.

All basis changes, evaluations, interpolation and reductions are linear over GF(2), so they compile to CNOTs. The code discovers the field embeddings from the **actual competition modulus** and derives every parity mask. The 144 basis-product assertions certify the complete bilinear tensor, not a random sample.

In [ ]:
"""Exact 42-product bilinear multiplier for GF(2^12). Apache-2.0."""
from functools import lru_cache


def gf_mul(a, b, modulus):
    out = 0
    n = modulus.bit_length() - 1
    while b:
        if b & 1:
            out ^= a
        b >>= 1
        a <<= 1
        if a >> n:
            a ^= modulus
    return out


def gf_pow(a, exponent, modulus):
    out = 1
    while exponent:
        if exponent & 1:
            out = gf_mul(out, a, modulus)
        a = gf_mul(a, a, modulus)
        exponent >>= 1
    return out


def xor_sum(values):
    out = 0
    for v in values:
        out ^= v
    return out


def apply_binary(columns, value):
    return xor_sum(c for j, c in enumerate(columns) if (value >> j) & 1)


def inverse_binary(columns):
    n = len(columns)
    rows = [sum(((c >> i) & 1) << j for j, c in enumerate(columns)) | (1 << (n+i)) for i in range(n)]
    for col in range(n):
        pivot = next(i for i in range(col, n) if (rows[i] >> col) & 1)
        rows[col], rows[pivot] = rows[pivot], rows[col]
        for i in range(n):
            if i != col and (rows[i] >> col) & 1:
                rows[i] ^= rows[col]
    return [sum(((rows[i] >> (n+j)) & 1) << i for i in range(n)) for j in range(n)]


def inverse_gf8(matrix):
    n = len(matrix)
    rows = [list(row) + [int(i == j) for j in range(n)] for i, row in enumerate(matrix)]
    for col in range(n):
        pivot = next(i for i in range(col, n) if rows[i][col])
        rows[col], rows[pivot] = rows[pivot], rows[col]
        scale = gf_pow(rows[col][col], 6, 0xB)
        rows[col] = [gf_mul(v, scale, 0xB) for v in rows[col]]
        for i in range(n):
            if i != col:
                scale = rows[i][col]
                rows[i] = [v ^ gf_mul(scale, w, 0xB) for v, w in zip(rows[i], rows[col])]
    return [row[n:] for row in rows]


def reduce_binary(poly, modulus):
    degree = modulus.bit_length() - 1
    while poly.bit_length() - 1 >= degree:
        poly ^= modulus << (poly.bit_length() - 1 - degree)
    return poly


@lru_cache(None)
def tower42(modulus=0x1053):
    """Return (left parity, right parity, output mask) for 42 AND terms.

    GF(8)[v]/(v^4+v+1), with GF(8)=GF(2)[u]/(u^3+u+1).
    Six finite evaluation points and infinity give seven GF(8) products.
    Each GF(8) product uses three diagonal and three cross products.
    """
    mul = lambda a, b: gf_mul(a, b, modulus)
    u = next(a for a in range(2, 4096) if gf_pow(a, 3, modulus) ^ a ^ 1 == 0)
    v = next(a for a in range(2, 4096) if gf_pow(a, 4, modulus) ^ a ^ 1 == 0)
    basis = [mul(gf_pow(v, k, modulus), gf_pow(u, j, modulus)) for k in range(4) for j in range(3)]
    to_tower = inverse_binary(basis)
    assert all(apply_binary(to_tower, b) == 1 << j for j, b in enumerate(basis))
    points = [0, 1, 2, 3, 4, 5]
    matrix = [[gf_pow(t, j, 0xB) for j in range(7)] for t in points] + [[0, 0, 0, 0, 0, 0, 1]]
    interpolation = inverse_gf8(matrix)
    base = []
    for i in range(3):
        output = (1 << (2*i)) ^ xor_sum(1 << (i+j) for j in range(3) if j != i)
        base.append((1 << i, 1 << i, reduce_binary(output,0xB)))
    for i in range(3):
        for j in range(i+1,3):
            mask = (1 << i) | (1 << j)
            base.append((mask,mask,reduce_binary(1 << (i+j),0xB)))
    result = []
    for point in range(7):
        def evaluate_input(x):
            coeffs = [(apply_binary(to_tower, x) >> (3*k)) & 7 for k in range(4)]
            if point == 6:
                return coeffs[3]
            return xor_sum(gf_mul(c, gf_pow(points[point], k, 0xB), 0xB) for k, c in enumerate(coeffs))
        eval_columns = [evaluate_input(1 << j) for j in range(12)]
        forms = [sum(((eval_columns[j] >> k) & 1) << j for j in range(12)) for k in range(3)]
        def lift_output(value):
            c = [gf_mul(interpolation[k][point], value, 0xB) for k in range(7)]
            reduced = (c[0]^c[4]) | ((c[1]^c[4]^c[5]) << 3) | ((c[2]^c[5]^c[6]) << 6) | ((c[3]^c[6]) << 9)
            return apply_binary(basis, reduced)
        for a,b,c in base:
            result.append((apply_binary(forms,a),apply_binary(forms,b),lift_output(c)))
    assert len(result) == 42
    for i in range(12):
        for j in range(12):
            actual = xor_sum(c for a,b,c in result if (a >> i)&1 and (b >> j)&1)
            assert actual == mul(1 << i,1 << j),(i,j)
    return tuple(result)



terms = tower42(field.modulus)
assert len(terms) == 42
assert all(xor_sum(c for a,b,c in terms if (a>>i)&1 and (b>>j)&1)
           == field.mul(1<<i,1<<j) for i in range(12) for j in range(12))
print('All 144 basis products agree with Kickmix. Bilinear rank:', len(terms))

## 3. Exhaust all 16,777,216 input pairs

The tensor certificate above is sufficient algebraically. This independent vectorized check also exercises all integer inputs against polynomial multiplication and reduction. It uses small batches rather than materializing a giant multiplication table.

In [ ]:
start=time.monotonic()
xs=np.arange(4096,dtype=np.uint16)
parity_table=np.array([i.bit_count()&1 for i in range(4096)],dtype=np.uint8)
checked=0
for lo in range(0,4096,64):
    av,bv=xs[lo:lo+64,None],xs[None,:]
    actual=np.zeros((64,4096),dtype=np.uint16)
    for am,bm,cm in terms:
        actual ^= (parity_table[av&am]&parity_table[bv&bm]).astype(np.uint16)*cm
    expected=np.zeros_like(actual)
    aa=np.broadcast_to(av,actual.shape).copy()
    bb=np.broadcast_to(bv,actual.shape).copy()
    for _ in range(12):
        expected ^= aa*(bb&1)
        bb >>= 1
        aa <<= 1
        aa ^= (aa>>12)*field.modulus
    assert np.array_equal(actual,expected),lo
    checked += actual.size
certificate={'modulus':field.modulus,'rank':len(terms),'basis_products':144,
             'exhaustive_pairs':checked,'seconds':time.monotonic()-start,
             'terms':[list(t) for t in terms]}
(OUT/'arithmetic_certificate.json').write_text(json.dumps(certificate,indent=2))
print(f'{checked:,} / {4096**2:,} pairs passed in {certificate["seconds"]:.2f}s')

## 4. Compile multiply-accumulate with two parity wires

A tensor term is $(a\cdot x)(b\cdot y)c$. Compute the two input parities, conjugate one Toffoli by a CNOT fanout corresponding to $c$, and update the next parity by XORing only the changed mask bits. A deterministic nearest-neighbor ordering cuts the parity-update CNOTs without changing the tensor.

For a quantum-controlled multiplication, first AND the control into one 12-bit input. This costs 12 additional Toffolis. Measurement-based uncomputation then removes that input mask for zero additional Toffolis: **54 versus the reference 75** for this primitive. The adapter delegates all unrelated builder operations to Kickmix.

In [ ]:
"""Kickmix adapter for exact GF(4096) multiply-accumulate. Apache-2.0."""
def order_terms(terms):
    remaining, ordered, previous = list(terms), [], (0, 0, 0)
    while remaining:
        index = min(range(len(remaining)), key=lambda i: (
            (remaining[i][0] ^ previous[0]).bit_count()
            + (remaining[i][1] ^ previous[1]).bit_count(), remaining[i]))
        previous = remaining.pop(index)
        ordered.append(previous)
    return tuple(ordered)


class TowerBuilder:
    def __init__(self, builder, km, terms):
        self.raw = builder
        self.km = km
        self.modulus = km.GF2Field(12).modulus
        # Keep parity accumulators live; order terms by nearest parity masks.
        self.terms = order_terms(terms)
        self.replacements = 0

    def __getattr__(self, name):
        return getattr(self.raw, name)

    def ixor_gf2_mul(self, lhs, rhs, *, target, field, control=True, tensor=None):
        b, km = self.raw, self.km
        if field.degree != 12 or field.modulus != self.modulus or isinstance(lhs, int) or isinstance(rhs, int):
            return b.ixor_gf2_mul(lhs, rhs, target=target, field=field, control=control)
        if control is False:
            return
        assert len(lhs) == len(rhs) == len(target) == 12
        # Target must not alias either input. The reference EEA meets this contract.
        assert not (set(target) & (set(lhs) | set(rhs)))
        masked = None
        if control is not True:
            masked = b.alloc_qubits(12)
            b.init_and(control, lhs, masked)
            original_lhs, lhs = lhs, masked
        scratch = b.alloc_qubits(2)
        previous_a = previous_b = 0
        for am, bm, cm in (self.terms if tensor is None else tensor):
            left = km.array([lhs[j] for j in range(12) if ((am ^ previous_a) >> j) & 1])
            right = km.array([rhs[j] for j in range(12) if ((bm ^ previous_b) >> j) & 1])
            b.cx(left, scratch[0])
            b.cx(right, scratch[1])
            # Conjugating one CCX by this CNOT fanout toggles exactly cm.
            pivot = (cm & -cm).bit_length()-1
            others = km.array([target[j] for j in range(12) if j != pivot and (cm >> j) & 1])
            if len(others):
                b.cx(target[pivot], others)
            b.ccx(scratch[0], scratch[1], target[pivot])
            if len(others):
                b.cx(target[pivot], others)
            previous_a, previous_b = am, bm
        b.cx(km.array([rhs[j] for j in range(12) if (previous_b >> j) & 1]), scratch[1])
        b.cx(km.array([lhs[j] for j in range(12) if (previous_a >> j) & 1]), scratch[0])
        b.free(scratch)
        if masked is not None:
            b.del_and(control, original_lhs, masked)
            b.free(masked)
        self.replacements += 1

## 5. Carry the saving through inversion and division

Let $\beta_k=x^{2^k-1}$. The addition chain $1,2,3,5,10,11$ needs five multiplications; one last squaring gives $x^{4094}=x^{-1}$ for nonzero $x$. As in the reference arithmetic, zero maps to zero. Squaring and Frobenius powers are linear maps. The adapter folds the left Frobenius map into the tensor parity masks, avoiding a separate 12-qubit register. The final squaring is synthesized in place.

The intermediate products remain live until cleanup. Measuring a product register with outcome $m$ contributes the sign $(-1)^{m\cdot\mathrm{product}}$. For each tensor term, a classically controlled CZ on its two input parities cancels that sign. Thus cleanup uses no Toffolis. A reused classical parity bit is explicitly reset before accumulating measurement outcomes.

Keeping this phase correction is essential: correct classical values and zero scratch bits alone do not certify the quantum operation. The next section checks the phase too.

In [ ]:
"""Measurement-uncomputed tower-field inverses for the DQI builder."""
class FullTowerBuilder(TowerBuilder):
    def __init__(self, builder, km, terms, field):
        super().__init__(builder, km, terms)
        self.field = field
        self.inverse_records = {}
        self.division_records = {}
        self.inverse_replacements = 0
        self.division_replacements = 0
        self.linear_ops = {}
        self.frobenius_tensors = {}

    def frobenius_terms(self, power):
        power %= 12
        if power not in self.frobenius_tensors:
            columns = [self.field.pow(1 << j, 1 << power) for j in range(12)]
            transformed = [(sum(((a & c).bit_count() & 1) << j for j,c in enumerate(columns)),b,out)
                           for a,b,out in self.terms]
            self.frobenius_tensors[power] = order_terms(transformed)
        return self.frobenius_tensors[power]

    def linear_inplace(self, target, power):
        power %= 12
        if power not in self.linear_ops:
            columns = [self.field.pow(1 << j, 1 << power) for j in range(12)]
            rows = [sum(((c >> i) & 1) << j for j,c in enumerate(columns)) for i in range(12)]
            ops = []
            for col in range(12):
                pivot = next(i for i in range(col,12) if (rows[i] >> col)&1)
                if pivot != col:
                    rows[col], rows[pivot] = rows[pivot], rows[col]
                    ops.append(('swap', col, pivot))
                for i in range(12):
                    if i != col and (rows[i] >> col)&1:
                        rows[i] ^= rows[col]
                        ops.append(('cx', col, i))
            self.linear_ops[power] = tuple(reversed(ops))
        for name, i, j in self.linear_ops[power]:
            getattr(self.raw,name)(target[i],target[j])

    def del_product(self, lhs, rhs, target, tensor=None):
        b, km = self.raw, self.km
        measurements = b.alloc_bits(12)
        parity = b.alloc_bits(1)[0]
        b.bit_store(parity, False)
        b.hmr(target, measurements)
        scratch = b.alloc_qubits(2)
        previous_a = previous_b = 0
        for am,bm,cm in (self.terms if tensor is None else tensor):
            left = km.array([lhs[j] for j in range(12) if ((am ^ previous_a) >> j)&1])
            right = km.array([rhs[j] for j in range(12) if ((bm ^ previous_b) >> j)&1])
            b.cx(left,scratch[0])
            b.cx(right,scratch[1])
            for j in range(12):
                if (cm >> j)&1:
                    b.cx(measurements[j],parity)
            b.push_condition(parity)
            b.cz(scratch[0],scratch[1])
            b.pop_condition()
            for j in range(12):
                if (cm >> j)&1:
                    b.cx(measurements[j],parity)
            previous_a, previous_b = am, bm
        b.cx(km.array([rhs[j] for j in range(12) if (previous_b >> j)&1]),scratch[1])
        b.cx(km.array([lhs[j] for j in range(12) if (previous_a >> j)&1]),scratch[0])
        b.free(scratch)
        b.free(measurements + km.array([parity]))

    def init_gf2_inverse_with_scaffold(self, input, *, target='alloc', field=None):
        field = self.field if field is None else field
        if field.degree != 12 or field.modulus != self.field.modulus:
            return self.raw.init_gf2_inverse_with_scaffold(input,target=target,field=field)
        b, km = self.raw, self.km
        if isinstance(target,str):
            assert target == 'alloc'
            target = b.alloc_qubits(12)
        intermediates = [b.alloc_qubits(12) for _ in range(4)]
        b2,b3,b5,b10 = intermediates
        # beta_k = x^(2^k-1): addition chain 1,2,3,5,10,11.
        steps = [(input,1,input,b2),(b2,1,input,b3),(b3,2,b2,b5),
                 (b5,5,b5,b10),(b10,1,input,target)]
        for src,power,rhs,out in steps:
            self.ixor_gf2_mul(src,rhs,target=out,field=field,tensor=self.frobenius_terms(power))
        self.linear_inplace(target,1)
        scaffold = km.array([q for reg in intermediates for q in reg])
        self.inverse_records[tuple(target)] = steps
        self.inverse_replacements += 1
        return target, scaffold

    def del_gf2_inverse_with_scaffold(self, input, *, target, scaffold, field=None):
        steps = self.inverse_records.pop(tuple(target),None)
        if steps is None:
            return self.raw.del_gf2_inverse_with_scaffold(input,target=target,scaffold=scaffold,field=field)
        b = self.raw
        self.linear_inplace(target,11)
        for src,power,rhs,out in reversed(steps):
            self.del_product(src,rhs,out,tensor=self.frobenius_terms(power))

    def init_gf2_div_with_scaffold(self, lhs, rhs, *, target='alloc', field=None):
        field = self.field if field is None else field
        if field.degree != 12 or field.modulus != self.field.modulus:
            return self.raw.init_gf2_div_with_scaffold(lhs,rhs,target=target,field=field)
        b,km = self.raw,self.km
        if isinstance(target,str):
            assert target == 'alloc'
            target = b.alloc_qubits(12)
        inverse, chain = self.init_gf2_inverse_with_scaffold(rhs,field=field)
        self.ixor_gf2_mul(lhs,inverse,target=target,field=field)
        self.division_records[tuple(target)] = (inverse,chain)
        self.division_replacements += 1
        return target, chain + inverse

    def del_gf2_div_with_scaffold(self, lhs, rhs, *, target, scaffold, field=None):
        record = self.division_records.pop(tuple(target),None)
        if record is None:
            return self.raw.del_gf2_div_with_scaffold(lhs,rhs,target=target,scaffold=scaffold,field=field)
        inverse,chain = record
        self.del_product(lhs,inverse,target)
        self.del_gf2_inverse_with_scaffold(rhs,target=inverse,scaffold=chain,field=field)

## 6. Test the emitted circuits, including measurement branches

The controlled multiply test uses arbitrary nonzero accumulator values and both control states. The inversion test covers every field element, including zero. Division uses every numerator and independently sampled denominators, explicitly including zero. Every test checks restored inputs, output values, zero quantum scratch, and no residual phase over 16 independent simulator seeds.

In [ ]:
primitive_rows=[]
def micro(custom):
    b = km.CircuitBuilder()
    a = b.create_quantum_register(12, name='a')
    x = b.create_quantum_register(12, name='b')
    c = b.create_quantum_register(12, name='out')
    ctrl = b.create_quantum_register(1, name='ctrl')[0]
    p = TowerBuilder(b, km, terms) if custom else b
    p.ixor_gf2_mul(a, x, target=c, control=ctrl, field=field)
    circ = b.finish_circuit()
    return circ, a, x, c, ctrl

rng = np.random.default_rng(20261023)
for custom in [False, True]:
    circ, a, x, out, ctrl = micro(custom)
    primitive_rows.append({'primitive':'controlled multiply','variant':'tower42' if custom else 'reference','qubits':circ.num_qubits,'toffolis':circ.operation_statistics().num_toffoli_like,'instructions':len(circ)})
    for seed in range(16):
        aa, bb, cc = rng.integers(0,4096,size=(3,256))
        controls = rng.integers(0,2,size=256)
        sim = km.Simulator(batch_size=256,seed=seed,debug_print=False)
        sim.use_same_registers_as(circ)
        bits = lambda v: ((np.asarray(v)[None,:] >> np.arange(12)[:,None])&1).astype(bool)
        for reg, data in [(a,aa),(x,bb),(out,cc)]:
            sim.write_across_shots(reg,bits(data))
        sim.write_across_shots(km.array([ctrl]), controls[None,:].astype(bool))
        sim.do(circ)
        expected = cc ^ (np.array([field.mul(int(av),int(bv)) for av,bv in zip(aa,bb)])*controls)
        assert np.array_equal(sim.read_across_shots(out),bits(expected))
        assert np.array_equal(sim.read_across_shots(a),bits(aa))
        assert np.array_equal(sim.read_across_shots(x),bits(bb))
        assert np.array_equal(sim.read_across_shots(km.array([ctrl])),controls[None,:].astype(bool))
        assert not sim.read_phase_flipped_across_shots().any()
        used=set(a)|set(x)|set(out)|{ctrl}
        # Scratch was allocated after the 37 interface qubits.
        if circ.num_qubits>37:
            assert not sim.read_across_shots(km.array([km.q(j) for j in range(37,circ.num_qubits)])).any()
    print('Controlled multiply passed:', 'tower42' if custom else 'reference')


def micro2(kind, custom):
    raw=km.CircuitBuilder()
    b=FullTowerBuilder(raw,km,terms,field) if custom else raw
    a=b.create_quantum_register(12,name='a')
    den=b.create_quantum_register(12,name='den')
    output=b.create_quantum_register(12,name='output')
    if kind=='inverse':
        target,scaffold=b.init_gf2_inverse_with_scaffold(a,field=field)
        b.cx(target,output)
        b.del_gf2_inverse_with_scaffold(a,target=target,scaffold=scaffold,field=field)
    else:
        target,scaffold=b.init_gf2_div_with_scaffold(a,den,field=field)
        b.cx(target,output)
        b.del_gf2_div_with_scaffold(a,den,target=target,scaffold=scaffold,field=field)
    b.free(target+scaffold)
    circ=b.finish_circuit()
    primitive_rows.append({'primitive':kind+' + cleanup','variant':'tower42' if custom else 'reference','qubits':circ.num_qubits,'toffolis':circ.operation_statistics().num_toffoli_like,'instructions':len(circ)})
    for seed in range(16):
        aa=np.arange(seed*256,(seed+1)*256)
        dd=np.random.default_rng(seed).integers(0,4096,size=256)
        dd[0]=0
        expected=np.array([field.invert(int(v)) if v else 0 for v in aa]) if kind=='inverse' else np.array([field.div(int(av),int(dv)) if dv else 0 for av,dv in zip(aa,dd)])
        bits=lambda v: ((v[None,:]>>np.arange(12)[:,None])&1).astype(bool)
        sim=km.Simulator(batch_size=256,seed=seed,debug_print=False)
        sim.use_same_registers_as(circ)
        sim.write_across_shots(a,bits(aa))
        sim.write_across_shots(den,bits(dd))
        sim.do(circ)
        assert np.array_equal(sim.read_across_shots(a),bits(aa))
        assert np.array_equal(sim.read_across_shots(den),bits(dd))
        assert np.array_equal(sim.read_across_shots(output),bits(expected)),(kind,custom,seed)
        assert not sim.read_across_shots(km.array([km.q(j) for j in range(36,circ.num_qubits)])).any(),(kind,'junk')
        assert not sim.read_phase_flipped_across_shots().any(),(kind,'phase',custom,seed)
    print(kind+' passed:', 'tower42' if custom else 'reference')

for kind in ['inverse','division']:
    for custom in [False,True]: micro2(kind,custom)


primitive_df=pd.DataFrame(primitive_rows)
display(primitive_df)
primitive_df.to_csv(OUT/'primitive_resources.csv',index=False)

## 7. Build the full decoder

This calls the supplied `ProosAndZalkaDecoder` with the official 222-iteration bound. Only multiplication, scaffolded inversion and scaffolded division are replaced. The reference circuit is read directly from the official sample submission.

Primitive scratch widths can increase even when the complete decoder's peak width does not. The next table measures the peak width of the whole circuit, which is what the competition uses.

In [ ]:
start=time.monotonic()
raw=km.CircuitBuilder()
builder=FullTowerBuilder(raw,km,terms,field)
syndromes=[builder.create_quantum_register(12,name=f'S{i}') for i in range(70)]
mbuc=[builder.create_classical_register(12,name=f'E{j}') for j in range(4095)]
iterations=reference.default_n_iters(70)
reference.ProosAndZalkaDecoder(70,field).build_syndrome_decoding_mbuc(
    builder,syndromes=syndromes,error_mbuc=mbuc,n_iters=iterations)
candidate=builder.finish_circuit()
assert not builder.inverse_records and not builder.division_records
candidate_path=SCRATCH/'dqi.kmb'
candidate.to_file(candidate_path,format='kmb')
build_info={'seconds':time.monotonic()-start,'iterations':iterations,
            'multiply_replacements':builder.replacements,
            'inverse_replacements':builder.inverse_replacements,
            'division_replacements':builder.division_replacements}
print(json.dumps(build_info,indent=2))
del builder,raw
baseline=km.Circuit.from_file(SAMPLE/'dqi.kmb',format='kmb')
resource_df=pd.DataFrame([
    {'variant':name,'qubits':c.num_qubits,'max_toffolis':c.operation_statistics().num_toffoli_like,
     'instructions':len(c),'reaction_depth':c.reaction_depth()}
    for name,c in [('reference',baseline),('tower42',candidate)]])
display(resource_df)
resource_df.to_csv(OUT/'decoder_resources.csv',index=False)

## 8. Run the official verifier at full strength

This runs the unchanged competition verifier with its usual $2^{-128}$ confidence parameter, full benchmark dimensions, and all 102,400 shots. Inputs are derived from each circuit's hash plus the public validation salt. The leaderboard will use its own secret salt.

The reported $T$ is the mean executed Toffoli-equivalent count, including classically conditional execution. We retain the official failure-rate bound and repetition overhead. A reduced-shot experiment would have a different confidence penalty and is not a substitute for these results.

In [ ]:
spec=official.ProblemSpecDQI.benchmark_instance(num_shots=SHOTS)
verification={}
for name,circuit in [('reference',baseline),('tower42',candidate)]:
    start=time.monotonic()
    print(f'Verifying {name}: {SHOTS:,} shots...',flush=True)
    summary=official.verify(circuit,spec,salt=PUBLIC_SALT,max_workers=WORKERS)
    summary['verification_seconds']=time.monotonic()-start
    verification[name]=summary
    (OUT/'verification.json').write_text(json.dumps(verification,indent=2))
    print(json.dumps({k:summary[k] for k in ['status','verified_shots','value_errors',
         'phase_errors','num_qubits','toffoli_count','score','verification_seconds']},indent=2),flush=True)
    assert summary['status']=='PASS'
    assert summary['value_errors']==summary['phase_errors']==0
ref_cost=verification['reference']['score']
new_cost=verification['tower42']['score']
improvement=ref_cost/new_cost
result_df=pd.DataFrame([
    {'variant':name,'Q':s['num_qubits'],'mean T':s['toffoli_count'],
     'value errors':s['value_errors'],'phase errors':s['phase_errors'],
     'epsilon upper':s['effective_error_rate'],'R':s['repetition_factor'],'DQI cost':s['score']}
    for name,s in verification.items()])
display(result_df)
print(f'DQI cost reduction: {100*(1-new_cost/ref_cost):.2f}%')
print(f'DQI improvement factor: {improvement:.6f}')
print(f'Projected three-task score with reference invsqrt and chemistry: {improvement**(1/3):.6f}')
print('Projection from local verification; no leaderboard submission is made by this notebook.')

## 9. Read the trade-off

The left panel measures the complete decoder, including its peak workspace. The right panel exposes the instruction-count trade-off: linear algebra and phase corrections are not physically free, even though the competition's Toffoli metric does not charge for CNOTs. Reaction depth and verification time are also included in the reports above.

In [ ]:
plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(1,2,figsize=(12,4.2),layout='constrained')
labels=['Reference','Tower42']; colors=['#8696aa','#167d9a']
toff=[verification[n]['toffoli_count']/1e6 for n in ['reference','tower42']]
bars=ax[0].bar(labels,toff,color=colors,width=.58)
ax[0].set(ylabel='Mean Toffoli equivalents (millions)',title='Full DQI decoder: counted work')
for bar,value,q in zip(bars,toff,resource_df.qubits):
    ax[0].text(bar.get_x()+bar.get_width()/2,value,f'{value:.3f}M'+chr(10)+f'{q:,} qubits',ha='center',va='bottom')
ax[0].set_ylim(0,max(toff)*1.24)
ops=resource_df.instructions.to_numpy()/1e6
bars=ax[1].bar(labels,ops,color=colors,width=.58)
ax[1].set(ylabel='Circuit instructions (millions)',title='The additional Clifford/control work')
for bar,value in zip(bars,ops):
    ax[1].text(bar.get_x()+bar.get_width()/2,value,f'{value:.1f}M',ha='center',va='bottom')
ax[1].set_ylim(0,max(ops)*1.18)
fig.savefig(OUT/'decoder_comparison.png',dpi=180,bbox_inches='tight')
plt.show()

## 10. Package a complete submission and record its identity

The archive contains the newly verified `dqi.kmb` plus the official sample `invsqrt.kmb` and `dfthcsos/` files. There is no CSV prediction task here. Archive paths are relative to the submission root.

The DQI candidate receives full validation above. The other two tasks are copied byte for byte from the supplied reference submission; their separate verifiers are not rerun in this notebook. An archive manifest records SHA-256 hashes, input-source identities and the arithmetic certificate.

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8<<20),b''): h.update(block)
    return h.hexdigest()

members=[(candidate_path,'dqi.kmb'),(SAMPLE/'invsqrt.kmb','invsqrt.kmb')]
members += [(p,p.relative_to(SAMPLE).as_posix()) for p in sorted((SAMPLE/'dfthcsos').rglob('*')) if p.is_file()]
assert all(p.suffix in {'.kmb','.kmx','.npz'} for p,_ in members)
archive_path=OUT/'submission.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for path,name in members: z.write(path,name)
with zipfile.ZipFile(archive_path) as z:
    assert z.testzip() is None
    assert len(z.namelist())==len(set(z.namelist()))
    assert {'dqi.kmb','invsqrt.kmb','dfthcsos/submission.npz'} <= set(z.namelist())
manifest={
    'archive_sha256':sha256_file(archive_path),
    'members':{name:sha256_file(path) for path,name in members},
    'source_sha256':{str(p.relative_to(COMPETITION)):sha256_file(p) for p in [
        VERIFIERS/'dqi'/'dqi_reference_circuit.py',VERIFIERS/'dqi'/'verify_submission_dqi.py']},
    'build':build_info,'shots_per_circuit':SHOTS,'validation_salt':PUBLIC_SALT,
    'dqi_improvement_factor':improvement,'projected_three_task_score':improvement**(1/3),
    'leaderboard_score':None,'license':'Apache-2.0',
}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
print(f'Archive: {archive_path.name}, {archive_path.stat().st_size/(1<<20):.1f} MiB')
print('SHA-256:',manifest['archive_sha256'])
display(FileLink(archive_path.name))

## What this result establishes, and what remains open

* The 42-term arithmetic matches all 144 basis products and every one of the $2^{24}$ input pairs.
* Emitted controlled multiplication, inversion and division restore their inputs and clean quantum workspace, with measurement-induced phases checked explicitly.
* The complete decoder keeps the original algorithm and iteration bound; its saving is measured under the official verifier.
* The three-task score is a projection from public-salt tests, not an observed leaderboard score. Sampled full-circuit verification is not an exhaustive proof of the entire decoder.

Next research directions include lower-rank GF(4096) multiplication, better linear-map synthesis, and decoder-level changes that reduce workspace or synchronized EEA work. Any approximation should be assessed against $R(\epsilon^+)Q^2T$ with full-strength verification. Fusion and chemistry offer separate opportunities; this notebook isolates one exact, reusable arithmetic improvement.

**Related public work.** [Quantum Circuits | Memory & Algebra](https://www.kaggle.com/code/lightningv08/quantum-circuits-memory-algebra) also integrates rank-42 arithmetic into the supplied decoder, using formulas from [Gu, Kemeny, Yelin and Lukin](https://arxiv.org/abs/2610.22367). Rank 42 itself is not a novelty claim here. This notebook derives its tensor from a GF(8) tower, folds Frobenius maps into input parities, schedules those parities, and runs fresh full-strength checks.

**Attribution and license.** The decoder algorithm, reference circuits, Kickmix and verifiers come from the official competition data and sources linked above. The new adapter and validation code are released under Apache 2.0. No claim is made that 42 is the minimum possible bilinear rank or that tower-field multiplication is new.